# XD2-Net — 01: Data Preparation

Loads an event log described by a dataset YAML, splits it into train/test, discovers or
reads roles, normalises and index-encodes features, and saves one artifact per
prefix-length bucket for the training notebook. The dynamic (per-event) input is built
in the selected model's own format: one-hot for `baseline`, packed field tokens for
`hierarchical`. Everything else is shared.

Run this once per (model, dataset) before training. `scripts/run_experiment.py` does
01–04 in one go.

In [ ]:
import os, sys

# ── Colab / local detection ──────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/xd2-net'
    %pip install -e {REPO_ROOT} --quiet
else:
    REPO_ROOT = os.path.abspath('..')
    sys.path.insert(0, REPO_ROOT)

os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

In [ ]:
# --- parameters --------------------------------------------------------------
MODEL = 'baseline'                                   # 'baseline' | 'hierarchical'
DATASET_CONFIG = 'configs/datasets/bpic2012.yaml'    # any dataset YAML (see configs/datasets/_template.yaml)
MODEL_CONFIG = None                                  # None = configs/models/<MODEL>.yaml

In [ ]:
import glob
import json
import logging
import pickle
import numpy as np
import pandas as pd

from xd2net.common.config import load_dataset_config, load_experiment_config
from xd2net.common.data import build_bucket, prepare_dataset, resolve_prefix_lengths
from xd2net.common.registry import get_spec

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

spec = get_spec(MODEL)
exp = load_experiment_config(MODEL_CONFIG or spec.default_config, spec.model_config_cls)
ds = load_dataset_config(DATASET_CONFIG)
run_dir = os.path.join(exp.workspace_dir, MODEL, ds.name)
artifacts_dir = os.path.join(run_dir, 'artifacts')
ckpt_dir = os.path.join(run_dir, 'checkpoints')
print('Model:', MODEL, '| dataset:', ds.name, '| log:', ds.path)

In [ ]:
# ── Raw event log ─────────────────────────────────────────────────────────────
df = pd.read_csv(ds.path) if ds.path.endswith('.csv') else pd.read_parquet(ds.path)
print(df.shape)
df.head()

In [ ]:
# ── Load, label, split, roles, impute, normalise, index (fit on train only) ──
prepared = prepare_dataset(ds, seed=exp.training.seed)
print(json.dumps(prepared.summary(), indent=2))
print('Roles:', prepared.train['ROLE'].value_counts().to_dict())
print('Outcome (train cases):', prepared.train.groupby('CASEID')['OUTCOME'].first().value_counts().to_dict())

In [ ]:
# ── Per-bucket inputs ─────────────────────────────────────────────────────────
prefix_lengths = resolve_prefix_lengths(prepared)
artifacts = {}  # keyed by bucket_key, e.g. '1_15'

for n_size in prefix_lengths:
    bucket = build_bucket(prepared, n_size, spec.encode_dynamic, seed=exp.training.seed)
    if bucket is None:
        print(f'1_{n_size}: skipped (too few cases or a single class)')
        continue
    artifacts[bucket['bucket_key']] = bucket
    print(f"{bucket['bucket_key']}: train={bucket['n_train']}  test={bucket['n_test']}  "
          f"dynamic={bucket['inputs_train']['input_dynamic'].shape[1:]}  exp_dim={bucket['exp_dim']}")

print('\nDone. Buckets prepared:', list(artifacts.keys()))

In [ ]:
# ── Save artifacts ────────────────────────────────────────────────────────────
os.makedirs(artifacts_dir, exist_ok=True)

for bucket_key, art in artifacts.items():
    with open(os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl'), 'wb') as f:
        pickle.dump(art, f)

# Also save indexes and scalers
with open(os.path.join(artifacts_dir, 'indexes.pkl'), 'wb') as f:
    pickle.dump(prepared.indexes, f)
with open(os.path.join(artifacts_dir, 'scalers.pkl'), 'wb') as f:
    pickle.dump(prepared.scalers, f)

print('Artifacts saved to', artifacts_dir)

In [ ]:
# ── Verification: reload and check shapes ────────────────────────────────────
for bucket_key in list(artifacts.keys())[:3]:
    with open(os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl'), 'rb') as f:
        a = pickle.load(f)
    print(f"{bucket_key}: dynamic={a['inputs_train']['input_dynamic'].shape}  "
          f"static={a['inputs_train']['input_static'].shape}  "
          f"exp={a['inputs_train']['input_exp'].shape}")